In [ ]:
# ===========================================
# Emotion IAA + LLM Alignment Evaluation (Colab)
# Works with:
#   - 3 human annotation files in .json
#   - 3 human annotation files in .jsonl
#   - 1 LLM annotation file in .jsonl
#
# Computes:
#   (1) Krippendorff's alpha (nominal) for humans on ALL 9 labels
#   (2) Krippendorff's alpha (nominal) for humans on EMOTION-ONLY 6 labels
#   (3) Human consensus labels + tie/no-consensus rate
#   (4) LLM vs human-consensus Macro F1 (ALL 9 + EMOTION-ONLY)
#   (5) Aspect-wise human alpha (ALL 9 labels)
#
# Assumes each line/object has structure:
#   {"input": "...", "output": [{"aspect": "...", "emotion": "..."}, ...]}
#
# IMPORTANT:
#   - review_id is determined by the row index (order) in each file
#   - all human files AND llm file must have the SAME number of items (e.g., 50)
# ===========================================

import json
from pathlib import Path
from collections import Counter
from typing import Dict, List, Tuple, Optional

import pandas as pd

from sklearn.metrics import f1_score, classification_report

# ----------------------------
# 0) CONFIG: your label sets
# ----------------------------
POS = {"admiration", "satisfaction", "gratitude"}
NEG = {"disappointment", "annoyance", "disgust"}
NEUTRAL_LIKE = {"neutral", "mixed_emotions", "mentioned_only"}

ALL_LABELS = sorted(POS | NEG | NEUTRAL_LIKE)          # 9 labels
EMOTION_ONLY_LABELS = sorted(POS | NEG)                # 6 labels (no neutral-like)

In [ ]:

import json
from pathlib import Path

def load_any_json(path: Path):
    """
    Robust loader:
    - If file starts with '[' or '{' and parses as JSON -> use json.load
    - If json.load fails with 'Extra data' -> treat as JSONL (one object per line)
    - If .jsonl -> treat as JSONL
    Returns list of objects in order.
    """
    suffix = path.suffix.lower()

    # Always treat .jsonl as JSONL
    if suffix == ".jsonl":
        rows = []
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        return rows

    # For .json (or anything else), try normal JSON first
    text = path.read_text(encoding="utf-8").strip()
    if not text:
        return []

    try:
        data = json.loads(text)
        # If it's a list, that's our rows
        if isinstance(data, list):
            return data
        # If it's a dict of objects, return values (sorted by numeric keys if possible)
        if isinstance(data, dict):
            keys = list(data.keys())
            if all(isinstance(k, str) and k.isdigit() for k in keys):
                return [data[str(i)] for i in sorted(map(int, keys))]
            return list(data.values())
        raise ValueError(f"Unsupported JSON top-level type in {path}: {type(data)}")

    except json.JSONDecodeError as e:
        # If normal JSON parsing fails, try JSONL
        rows = []
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        return rows



def file_to_long_df(path: Path, annotator_id: str) -> pd.DataFrame:
    """
    Convert one annotator file into a "long" dataframe:
      review_id, aspect, annotator, emotion

    review_id is assigned by item position in the file (0..N-1).
    This requires that all annotators used the same ordering of reviews.
    """
    data = load_any_json(path)
    out_rows = []

    for review_id, item in enumerate(data):
        outputs = item.get("output", [])
        for obj in outputs:
            aspect = obj.get("aspect", None)
            emotion = obj.get("emotion", None)
            if aspect is None or emotion is None:
                continue
            out_rows.append(
                {
                    "review_id": review_id,
                    "aspect": str(aspect).strip(),
                    "annotator": annotator_id,
                    "emotion": str(emotion).strip(),
                }
            )

    return pd.DataFrame(out_rows)


def load_annotators(file_paths: List[str]) -> pd.DataFrame:
    """
    Load multiple annotator files into a single dataframe.
    Each file becomes one annotator (annotator_id = filename stem).
    """
    dfs = []
    for fp in file_paths:
        p = Path(fp)
        dfs.append(file_to_long_df(p, annotator_id=p.stem))
    return pd.concat(dfs, ignore_index=True)


def check_review_counts(file_paths: List[str], expected: Optional[int] = None) -> None:
    """
    Check that each file contains the same number of items (e.g., 50).
    This is critical because we align by file order -> review_id.
    """
    counts = []
    for fp in file_paths:
        p = Path(fp)
        n = len(load_any_json(p))
        counts.append(n)
        print(f"{p.name}: {n} items")
    if len(set(counts)) != 1:
        raise ValueError(f"Mismatch in number of items across files: {counts}")
    if expected is not None and counts[0] != expected:
        print(f"WARNING: expected {expected} items but found {counts[0]}.")
    print("✅ All files have the same number of items.")


In [ ]:

# ---------------------------------------------------------
# 2) Krippendorff's alpha (nominal) implementation
# ---------------------------------------------------------
def build_ratings_by_unit(df: pd.DataFrame) -> Dict[Tuple[int, str], List[str]]:
    """
    Build dict mapping (review_id, aspect) -> list of emotion labels (one per rater).
    Missing values are naturally omitted because they won't appear in df.
    """
    grouped = df.groupby(["review_id", "aspect"])["emotion"].apply(list)
    return {k: v for k, v in grouped.items()}


def krippendorff_alpha_nominal(
    ratings_by_unit: Dict[Tuple[int, str], List[str]],
    allowed_labels: Optional[List[str]] = None,
) -> float:
    """
    Krippendorff's alpha for nominal categories.
    - Works with multiple raters
    - Handles missingness (units with <2 ratings contribute nothing)

    allowed_labels:
      If provided, uses a fixed label universe (recommended for stable reporting).
    """
    # Determine label list
    label_set = set()
    for labs in ratings_by_unit.values():
        for lab in labs:
            label_set.add(lab)

    labels = list(allowed_labels) if allowed_labels is not None else sorted(label_set)
    if len(labels) < 2:
        return float("nan")

    idx = {lab: i for i, lab in enumerate(labels)}
    L = len(labels)

    # Coincidence matrix O
    O = [[0.0 for _ in range(L)] for _ in range(L)]

    for _, labs in ratings_by_unit.items():
        # keep only labels in our universe
        labs = [lab for lab in labs if lab in idx]
        m = len(labs)
        if m < 2:
            continue

        counts = Counter(labs)
        denom = m - 1

        # Fill coincidence counts
        for lab_i, n_i in counts.items():
            i = idx[lab_i]
            # diagonal term
            O[i][i] += n_i * (n_i - 1) / denom
            # off-diagonals
            for lab_j, n_j in counts.items():
                if lab_j == lab_i:
                    continue
                j = idx[lab_j]
                O[i][j] += n_i * n_j / denom

    row_sums = [sum(row) for row in O]
    N = sum(row_sums)
    if N == 0:
        return float("nan")

    # Observed disagreement Do (nominal): delta=1 if i!=j else 0
    diag = sum(O[i][i] for i in range(L))
    Do = (N - diag) / N

    # Expected disagreement De from marginals
    if N <= 1:
        return float("nan")

    denom = N - 1
    diagE = sum((n * (n - 1)) / denom for n in row_sums)
    De = (N - diagE) / N

    if De == 0:
        return 1.0 if Do == 0 else float("nan")

    return 1 - (Do / De)

In [ ]:

# ---------------------------------------------------------
# 3) Human consensus + tie handling
# ---------------------------------------------------------
def majority_vote(labels: List[str]) -> Tuple[Optional[str], bool]:
    """
    Returns (consensus_label, tie_or_no_consensus).
    Tie includes:
      - empty list
      - multiple top labels with same max count
    """
    if not labels:
        return None, True
    c = Counter(labels)
    top_count = c.most_common(1)[0][1]
    top_labels = [lab for lab, cnt in c.items() if cnt == top_count]
    if len(top_labels) != 1:
        return None, True
    return top_labels[0], False


def consensus_table(human_df: pd.DataFrame) -> pd.DataFrame:
    """
    One row per (review_id, aspect):
      human_consensus, tie_or_no_consensus, n_ratings
    """
    rows = []
    for (rid, asp), sub in human_df.groupby(["review_id", "aspect"]):
        labs = sub["emotion"].tolist()
        cons, tie = majority_vote(labs)
        rows.append(
            {
                "review_id": rid,
                "aspect": asp,
                "human_consensus": cons,
                "tie_or_no_consensus": tie,
                "n_ratings": len(labs),
            }
        )
    return pd.DataFrame(rows)


In [ ]:

# ---------------------------------------------------------
# 4) LLM vs consensus metrics (Macro F1 + report)
# ---------------------------------------------------------
def load_llm_predictions(llm_path: str) -> pd.DataFrame:
    """
    Loads LLM annotations (jsonl or json also supported) and returns:
      review_id, aspect, llm_emotion

    If duplicates exist for (review_id, aspect), keeps the first.
    """
    llm_df = file_to_long_df(Path(llm_path), annotator_id="LLM")
    llm_df = llm_df.sort_values(["review_id", "aspect"]).drop_duplicates(["review_id", "aspect"], keep="first")
    return llm_df[["review_id", "aspect", "emotion"]].rename(columns={"emotion": "llm_emotion"})


def compute_llm_vs_consensus(cons_df: pd.DataFrame, llm_df: pd.DataFrame) -> None:
    """
    Compare LLM predictions to human consensus.
    Excludes tie/no-consensus items from reference.
    """
    merged = cons_df.merge(llm_df, on=["review_id", "aspect"], how="inner")

    # Remove items without consensus
    merged = merged[~merged["tie_or_no_consensus"]].copy()
    merged = merged.dropna(subset=["human_consensus", "llm_emotion"])

    print("\n=== LLM vs Human Consensus: ALL 9 labels ===")
    print("Items used:", len(merged))
    if len(merged) == 0:
        print("No comparable items found (check aspect matching / review alignment).")
        return

    y_true = merged["human_consensus"].tolist()
    y_pred = merged["llm_emotion"].tolist()

    macro_f1 = f1_score(y_true, y_pred, average="macro", labels=ALL_LABELS)
    print("Macro F1:", macro_f1)
    print(classification_report(y_true, y_pred, labels=ALL_LABELS, zero_division=0))

    # Emotion-only subset (true label in 6 emotion labels)
    emo = merged[merged["human_consensus"].isin(EMOTION_ONLY_LABELS)].copy()
    print("\n=== LLM vs Human Consensus: EMOTION-ONLY (6 labels) ===")
    print("Items used:", len(emo))
    if len(emo) > 0:
        y_true_e = emo["human_consensus"].tolist()
        y_pred_e = emo["llm_emotion"].tolist()
        macro_f1_e = f1_score(y_true_e, y_pred_e, average="macro", labels=EMOTION_ONLY_LABELS)
        print("Macro F1:", macro_f1_e)
        print(classification_report(y_true_e, y_pred_e, labels=EMOTION_ONLY_LABELS, zero_division=0))
    else:
        print("No emotion-only items after filtering (all consensus labels were neutral-like).")


In [ ]:

# ---------------------------------------------------------
# 5) Main runner (edit file paths here)
# ---------------------------------------------------------
def run_evaluation(
    human_files: List[str],
    llm_file: str,
    expected_n_reviews: Optional[int] = 50
):
    # --- 5.1 Validate that all files have same number of reviews ---
    print("Checking review counts (must match across all files)...")
    check_review_counts(human_files + [llm_file], expected=expected_n_reviews)

    # --- 5.2 Load humans ---
    human_df = load_annotators(human_files)

    # --- 5.3 Label sanity check ---
    unknown = set(human_df["emotion"].unique()) - set(ALL_LABELS)
    if unknown:
        print("\n⚠️ WARNING: Unknown human labels found (not in your 9-label set):", unknown)
        print("Fix typos/case/spaces or map them before computing metrics.\n")

    # --- 5.4 Human Krippendorff's alpha (ALL 9) ---
    ratings_all = build_ratings_by_unit(human_df)
    alpha_all = krippendorff_alpha_nominal(ratings_all, allowed_labels=ALL_LABELS)
    print("\n=== Krippendorff's alpha (Humans, ALL 9 labels) ===")
    print(alpha_all)

    # --- 5.5 Human Krippendorff's alpha (EMOTION-ONLY 6) ---
    human_emo_only = human_df[human_df["emotion"].isin(EMOTION_ONLY_LABELS)].copy()
    ratings_emo = build_ratings_by_unit(human_emo_only)
    alpha_emo = krippendorff_alpha_nominal(ratings_emo, allowed_labels=EMOTION_ONLY_LABELS)
    print("\n=== Krippendorff's alpha (Humans, EMOTION-ONLY 6 labels) ===")
    print(alpha_emo)

    # --- 5.6 Human consensus + tie rates ---
    cons = consensus_table(human_df)
    print("\n=== Human consensus stats ===")
    print("Units (review, aspect):", len(cons))
    print("Avg ratings per unit:", cons["n_ratings"].mean() if len(cons) else float("nan"))
    print("Tie/no-consensus rate:", cons["tie_or_no_consensus"].mean() if len(cons) else float("nan"))

    # --- 5.7 Aspect-wise human alpha (ALL 9) ---
    print("\n=== Aspect-wise Krippendorff's alpha (Humans, ALL 9 labels) ===")
    for asp, sub in human_df.groupby("aspect"):
        r = build_ratings_by_unit(sub)
        a = krippendorff_alpha_nominal(r, allowed_labels=ALL_LABELS)
        print(f"{asp:15s} alpha={a:.4f}")

    # --- 5.8 Load LLM and compute Macro F1 vs consensus ---
    llm_df = load_llm_predictions(llm_file)

    llm_unknown = set(llm_df["llm_emotion"].unique()) - set(ALL_LABELS)
    if llm_unknown:
        print("\n⚠️ WARNING: Unknown LLM labels found (not in your 9-label set):", llm_unknown)
        print("Consider mapping invalid labels to something like 'mixed_emotions' or fixing generation.\n")

    compute_llm_vs_consensus(cons, llm_df)

    # --- 5.9 Return dataframes in case you want to inspect them in Colab ---
    return human_df, cons, llm_df

In [ ]:

# =========================================================
# 6) EDIT THESE PATHS (Colab local files)
# ---------------------------------------------------------
# If you uploaded files via Colab "Files" sidebar, they will
# typically be in /content/<filename>
# =========================================================

human_files = [
    "/content/mustafa.json",
    "/content/mane.json",
    "/content/dani.jsonl",
    "/content/danila.jsonl",
    "/content/iuliia.jsonl",
    "/content/sama.jsonl",
]

llm_file = "/content/gpt.jsonl"

# Run everything
human_df, consensus_df, llm_df = run_evaluation(
    human_files=human_files,
    llm_file=llm_file,
    expected_n_reviews=50
)

Checking review counts (must match across all files)...
mustafa.json: 50 items
mane.json: 50 items
dani.jsonl: 50 items
danila.jsonl: 50 items
iuliia.jsonl: 50 items
sama.jsonl: 50 items
gpt.jsonl: 50 items
✅ All files have the same number of items.

=== Krippendorff's alpha (Humans, ALL 9 labels) ===
0.719632602292136

=== Krippendorff's alpha (Humans, EMOTION-ONLY 6 labels) ===
0.7539638245516121

=== Human consensus stats ===
Units (review, aspect): 116
Avg ratings per unit: 5.732758620689655
Tie/no-consensus rate: 0.034482758620689655

=== Aspect-wise Krippendorff's alpha (Humans, ALL 9 labels) ===
ambience        alpha=0.6025
food            alpha=0.6612
menu            alpha=0.5421
miscellaneous   alpha=1.0000
place           alpha=0.5735
price           alpha=0.5600
service         alpha=0.6499
staff           alpha=0.6265

=== LLM vs Human Consensus: ALL 9 labels ===
Items used: 107
Macro F1: 0.6248986040369742
                precision    recall  f1-score   support

    admira

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [ ]:
from pathlib import Path

print("Files in /content:")
for p in sorted(Path("/content").glob("*")):
    print(" -", p)

print("\nFiles in /content/sample_data:")
for p in sorted(Path("/content/sample_data").glob("*")):
    print(" -", p)


Files in /content:
 - /content/.config
 - /content/dani.jsonl
 - /content/danila.jsonl
 - /content/gpt.jsonl
 - /content/iuliia.jsonl
 - /content/mane.json
 - /content/mustafa.json
 - /content/sama.jsonl
 - /content/sample_data

Files in /content/sample_data:
 - /content/sample_data/README.md
 - /content/sample_data/anscombe.json
 - /content/sample_data/california_housing_test.csv
 - /content/sample_data/california_housing_train.csv
 - /content/sample_data/mnist_test.csv
 - /content/sample_data/mnist_train_small.csv
